In [1]:
import numpy as np
from sklearn import preprocessing
raw_csv_data=np.loadtxt('Audiobooks_data.csv',delimiter=',')
unscaled_inputs_all=raw_csv_data[:,1:-1]
targets_all=raw_csv_data[:,-1]

In [2]:
shuffled_inidices = np.arange(unscaled_inputs_all.shape[0])
np.random.shuffle(shuffled_inidices)

unscaled_inputs_all = unscaled_inputs_all[shuffled_inidices]
targets_all = targets_all[shuffled_inidices]


In [3]:
# Policzymy, ile jest targetów 1 (co oznacza, że klient wrócił do sklepu)
num_one_targets = int(np.sum(targets_all))

# Ustaw licznik dla targetów, które wynoszą 0
#(oznaczających, że klient nie wrócił do sklepu)
zero_targets_counter=0

#Chcemy utworzyć "zrównoważony" zbiór danych,
# więc będziemy musieli usunąć niektóe pary wejście/target
# by targetów 0 i 1 było mniej więcej tyle samo.
# Zadeklarujemy zmienną, która to zrobi:
indices_to_remove = []

# Policzymy liczbę targetów, które są równe 0.
# Gdy liczba targetów z 0 będzie równa liczbie targetów z 1,
# zaznaczymy pozostałe wpisy target 0 do usunięcia.
for i in range(targets_all.shape[0]):
    if targets_all[i] == 0:
        zero_targets_counter += 1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)
# Utwórzymy dwie nowe zmienne:
# - jedną, która będzie zawierać dane wejściowe
# - jedną, która będzie zawierać targety.
# Usuwamy wszystkie indeksy, które oznaczyliśmy jako "do usunięcia" w pętli powyżej.
unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors = np.delete(targets_all, indices_to_remove, axis=0)

In [4]:
scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

In [5]:
shuffled_inidices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_inidices)

In [6]:
shuffled_inputs = scaled_inputs[shuffled_inidices]
shuffled_targets = targets_equal_priors[shuffled_inidices]

samples_count= shuffled_inputs.shape[0]

train_samples_count=int(0.8*samples_count)
validation_samples_count=int(0.1*samples_count)
test_samples_count=samples_count-train_samples_count-validation_samples_count

train_inputs=shuffled_inputs[:train_samples_count]
train_targets=shuffled_targets[:train_samples_count]

validation_inputs=shuffled_inputs[train_samples_count:train_samples_count+validation_samples_count]
validation_targets=shuffled_targets[train_samples_count:train_samples_count+validation_samples_count]

test_inputs=shuffled_inputs[train_samples_count+validation_samples_count:]
test_targets=shuffled_targets[train_samples_count+validation_samples_count:]

In [7]:
print(np.sum(train_targets), train_samples_count, np.sum(train_targets) / train_samples_count)
print(np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print(np.sum(test_targets), test_samples_count, np.sum(test_targets) / test_samples_count)

1796.0 3579 0.5018161497625034
222.0 447 0.4966442953020134
219.0 448 0.4888392857142857


In [8]:
np.savez('Audiobooks_data_train', inputs=train_inputs, targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test', inputs=test_inputs, targets=test_targets)